In [1]:
!pip install -q transformers accelerate bitsandbytes

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.9/40.9 MB 20.6 MB/s eta 0:00:00


In [2]:
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch

model_name = "Qwen/Qwen2.5-1.5B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.float16,
    device_map="auto"
)

def chat(prompt):
    messages = [{"role": "user", "content": prompt}]
    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(text, return_tensors="pt").to(model.device)
    outputs = model.generate(**inputs, max_new_tokens=200)
    response = tokenizer.decode(outputs[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True)
    print(response)
chat("What is the difference between a star and a planet?")

config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

...

model.safetensors: reconstructing file:   0%|          |  0.00B / 3.09GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

A star is an astronomical object that generates its own light through nuclear fusion in its core, while a planet is a celestial body orbiting around a star or stellar remnant. Stars can range from small, dim red dwarfs to massive, bright supergiants, whereas planets are typically much smaller and less luminous than stars. Additionally, stars have their own magnetic fields and emit radiation across all parts of the electromagnetic spectrum, including visible light, while planets do not produce their own light but reflect the light from nearby stars.


In [3]:
def chat_temp(prompt, temperature=0.7):
    messages = [{"role": "user", "content": prompt}]
    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(text, return_tensors="pt").to(model.device)
    outputs = model.generate(
        **inputs,
        max_new_tokens=100,
        do_sample=True,
        temperature=temperature
    )
    response = tokenizer.decode(outputs[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True)
    print(f"Temperature {temperature}:\n{response}\n")

chat_temp("Write one creative sentence about the moon.", temperature=0.1)
chat_temp("Write one creative sentence about the moon.", temperature=1.2)

Temperature 0.1:
The full moon casts its silver glow across the night sky, beckoning us to wander through the darkness and seek solace in its serene beauty.

Temperature 1.2:
The moon whispers secrets in its endless silver robes to those who dare look up at night.



In [4]:
!pip install -q peft datasets

In [5]:
training_data = [
    {"question": "Write a function to check if a number is prime.",
     "answer": "```python\ndef is_prime(n):\n    if n < 2:\n        return False\n    for i in range(2, int(n**0.5) + 1):\n        if n % i == 0:\n            return False\n    return True\n```\n\n**Why:** we only check divisors up to the square root of n, since a larger factor would have a matching smaller one we'd already have found."},

    {"question": "How do I reverse a string in Python?",
     "answer": "```python\ndef reverse_string(s):\n    return s[::-1]\n```\n\n**Why:** the `[::-1]` slice steps backward through the whole string, which is Python's fastest built-in way to reverse a sequence."},

    {"question": "Write a function to find the maximum value in a list.",
     "answer": "```python\ndef find_max(nums):\n    if not nums:\n        return None\n    return max(nums)\n```\n\n**Why:** we check for an empty list first to avoid an error, then use Python's built-in `max()` instead of writing a manual loop, since it's already optimized."},

    {"question": "How do I remove duplicates from a list?",
     "answer": "```python\ndef remove_duplicates(items):\n    return list(set(items))\n```\n\n**Why:** converting to a `set` automatically drops duplicates since sets only allow unique values, then we convert back to a list."},

    {"question": "Write a function to check if a string is a palindrome.",
     "answer": "```python\ndef is_palindrome(s):\n    s = s.lower().replace(' ', '')\n    return s == s[::-1]\n```\n\n**Why:** we normalize case and remove spaces first so 'Was it a car' still counts, then compare the string to its reverse."},

    {"question": "How do I count word frequency in a sentence?",
     "answer": "```python\ndef word_count(sentence):\n    words = sentence.lower().split()\n    counts = {}\n    for w in words:\n        counts[w] = counts.get(w, 0) + 1\n    return counts\n```\n\n**Why:** `.get(w, 0)` safely handles new words by defaulting their count to 0 before adding 1, avoiding a KeyError."},

    {"question": "Write a function to swap two variables.",
     "answer": "```python\ndef swap(a, b):\n    return b, a\n```\n\n**Why:** Python lets you return multiple values as a tuple, so this swaps without needing a temporary variable."},

    {"question": "How do I flatten a nested list?",
     "answer": "```python\ndef flatten(nested):\n    return [item for sublist in nested for item in sublist]\n```\n\n**Why:** a nested list comprehension iterates the outer list, then the inner list, pulling every item into one flat list."},
]

print(f"Training examples: {len(training_data)}")

Training examples: 8


In [6]:
from datasets import Dataset

def format_example(example):
    messages = [
        {"role": "user", "content": example["question"]},
        {"role": "assistant", "content": example["answer"]}
    ]
    text = tokenizer.apply_chat_template(messages, tokenize=False)
    return {"text": text}

formatted_data = [format_example(ex) for ex in training_data]
dataset = Dataset.from_list(formatted_data)

print(dataset[0]["text"][:300])

<|im_start|>system
You are Qwen, created by Alibaba Cloud. You are a helpful assistant.<|im_end|>
<|im_start|>user
Write a function to check if a number is prime.<|im_end|>
<|im_start|>assistant
```python
def is_prime(n):
    if n < 2:
        return False
    for i in range(2, int(n**0.5) + 1):
   


In [7]:
from peft import LoraConfig, get_peft_model, TaskType

lora_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=8,
    lora_alpha=16,
    lora_dropout=0.05,
    target_modules=["q_proj", "v_proj"],
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

ImportError: Found an incompatible version of torchao. Found version 0.10.0, but only versions above 0.16.0 are supported

In [ ]:
!pip uninstall -y torchao -q

In [ ]:
lora_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=8,
    lora_alpha=16,
    lora_dropout=0.05,
    target_modules=["q_proj", "v_proj"],
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

In [ ]:
from transformers import TrainingArguments, Trainer, DataCollatorForLanguageModeling

def tokenize_function(examples):
    return tokenizer(examples["text"], truncation=True, max_length=512, padding="max_length")

tokenized_dataset = dataset.map(tokenize_function, batched=True)

training_args = TrainingArguments(
    output_dir="./coding_assistant_lora",
    num_train_epochs=3,
    per_device_train_batch_size=2,
    learning_rate=2e-4,
    logging_steps=1,
    save_strategy="no",
    report_to="none",
)

data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset,
    data_collator=data_collator,
)

trainer.train()

In [ ]:
def chat_finetuned(prompt):
    messages = [{"role": "user", "content": prompt}]
    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(text, return_tensors="pt").to(model.device)
    outputs = model.generate(**inputs, max_new_tokens=200, do_sample=False)
    response = tokenizer.decode(outputs[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True)
    print(response)

print("=== FINE-TUNED (adapter ON) ===")
chat_finetuned("Write a function to check if two strings are anagrams.")

print("\n=== BASE MODEL (adapter OFF) ===")
with model.disable_adapter():
    chat_finetuned("Write a function to check if two strings are anagrams.")

In [ ]:
print("=== FINE-TUNED on a TRAINING-LIKE question ===")
chat_finetuned("Write a function to check if a number is a perfect square.")

In [ ]:
training_args_2 = TrainingArguments(
    output_dir="./coding_assistant_lora_v2",
    num_train_epochs=15,
    per_device_train_batch_size=2,
    learning_rate=2e-4,
    logging_steps=1,
    save_strategy="no",
    report_to="none",
)

trainer2 = Trainer(
    model=model,
    args=training_args_2,
    train_dataset=tokenized_dataset,
    data_collator=data_collator,
)

trainer2.train()

In [ ]:
print("=== FINE-TUNED v2 on TRAINING-LIKE question ===")
chat_finetuned("Write a function to check if a number is a perfect square.")

print("\n=== FINE-TUNED v2 on the NEW anagram question ===")
chat_finetuned("Write a function to check if two strings are anagrams.")